In [1]:
RUN_FINAL = False

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import mlflow

print(sys.executable)
print(Path.cwd())
assert (Path.cwd() / "pyproject.toml").is_file()

d:\campaign-mini\.venv\Scripts\python.exe
d:\campaign-mini


In [2]:
from pathlib import Path
import mlflow
from mlflow import MlflowClient

from tracking import configure_tracking

context = configure_tracking()
experiment_name = "campaign-mini"

experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        artifact_location=context["artifact_uri"],
    )
else:
    if experiment.lifecycle_stage != "active":
        raise ValueError("The existing experiment is not active")

    if (
        experiment.artifact_location.rstrip("/")
        != context["artifact_uri"].rstrip("/")
    ):
        raise ValueError(
            "The existing experiment uses a different artifact location"
        )

    experiment_id = experiment.experiment_id

mlflow.set_experiment(experiment_id=experiment_id)

print("Experiment:", experiment_name)
print("Tracking URI:", context["tracking_uri"])
print("Artifact location:", context["artifact_uri"])

2026/10/19 23:37:24 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/19 23:37:24 INFO mlflow.store.db.utils: Updating database tables


Experiment: campaign-mini
Tracking URI: sqlite:///C:/Users/KISHAN PRAJAPATI/AppData/Local/campaign-mini-state/mlflow.db
Artifact location: file:///C:/Users/KISHAN%20PRAJAPATI/AppData/Local/campaign-mini-state/artifacts


In [3]:
with mlflow.start_run(
    run_name="setup-check",
    tags={"purpose": "setup_check"},
) as run:
    mlflow.log_param("check_kind", "round_trip")
    mlflow.log_metric("check_value", 1.0)
    mlflow.log_text(
        "Tracking round-trip succeeded.",
        "setup_note.txt",
    )

run_id = run.info.run_id

print("Saved run ID:", run_id)

Saved run ID: dd717f5ea4014bcb9fd2d445efe0ec26


In [4]:
client = MlflowClient(tracking_uri=context["tracking_uri"])
saved = client.get_run(run_id)
note_path = client.download_artifacts(run_id, "setup_note.txt")
note_text = Path(note_path).read_text(encoding="utf-8")

assert saved.info.status == "FINISHED"
assert saved.data.params["check_kind"] == "round_trip"
assert saved.data.metrics["check_value"] == 1.0
assert saved.data.tags["purpose"] == "setup_check"
assert note_text == "Tracking round-trip succeeded."

print("Run ID:", run_id)
print("Status:", saved.info.status)
print("Parameter:", saved.data.params["check_kind"])
print("Metric:", saved.data.metrics["check_value"])
print("Tag:", saved.data.tags["purpose"])
print("Artifact:", note_text)
print("All setup checks passed.")

Run ID: dd717f5ea4014bcb9fd2d445efe0ec26
Status: FINISHED
Parameter: round_trip
Metric: 1.0
Tag: setup_check
Artifact: Tracking round-trip succeeded.
All setup checks passed.
